Code created by: Lorena Espinosa, Johana Rátiva, Eduards Chipatecua 

Class: Procesamiento del Lenguaje Natural

University: Universidad de los Andes

Date: August 23, 2026

In [44]:
import copy
import xml.etree.ElementTree as ET
import re
import unicodedata
import math
import sys
import spacy
import pandas as pd
import numpy as np
import nltk
import import_ipynb # Funciona para importar las metricas desde el otro notebook
import Metricas_Evaluacion as metricas

from IPython.display import display
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from pathlib import Path
from rank_bm25 import BM25Okapi
from gensim import corpora, models, similarities

stemmer = PorterStemmer()
for recurso in ["punkt", "punkt_tab"]:
    nltk.download(recurso, quiet=True)

/home/lorena/python/NLP-ml/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/lorena/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


# Motores de Busqueda

## Procesamiento DataSet

### Lectura de documentos

el contenido empezaba con el titulo del documento, lo cual puede resultar en busquedas dobles en elementos donde solo esta una vez, por eso conviene filtrarlo desde la lectura

In [45]:
def read_naf_document(path):
    """ 
    Esta función lee un documento en formato NAF (Natural Annotation Format) desde la ruta especificada
    y extrae información relevante del mismo a partir de su estructura XML.
    Devuelve un diccionario con el 
        ID del documento, 
        el título y 
        el contenido del texto.

    Se limpia el contenido eliminando el título del mismo si este existe,
    y se eliminan los caracteres de puntuación y espacios en blanco al inicio del contenido.
    """
    tree = ET.parse(path) # abre el archivo NAF y lo parsea en un árbol XML
    root = tree.getroot() # obtiene la raíz del árbol XML 

    header = root.find("nafHeader") # acceso a los hijos del elemento raíz
    file_desc = header.find("fileDesc")
    public = header.find("public")
    raw = root.find("raw")
    titulo = file_desc.get("title")
    contenido = raw.text

    if titulo is not None:
        contenido = contenido[len(titulo):].lstrip(". \n")
        return {
            "id": public.get("publicId"),
            "titulo": titulo,
            "contenido": contenido}
    else:
        return {
            "id": public.get("publicId"),
            "contenido": contenido}

docs_path = Path("docs-raw-texts")
naf_files = list(docs_path.glob("*.naf"))
documents = []

for file_path in naf_files:
    document = read_naf_document(file_path)
    documents.append(document)

In [46]:
# Tabla de verificación de los documentos leídos
# Seleccionamos los mismos 10 documentos para las tablas del informe
sample_documents = documents[:10]

tabla_lectura = pd.DataFrame([
    {
        "id": document["id"],
        "titulo": document.get("titulo", ""),
        "longitud_contenido": len(document.get("contenido", ""))
    }
    for document in sample_documents
])

display(tabla_lectura)

,id,titulo,longitud_contenido
0,d096,Joseph Nicollet and the Upper Mississippi River,4502
1,d169,The very first Printed Book,4627
2,d027,Giovanni Maria Lancisi and his Medical Discove...,3884
3,d203,GNU’s not Unix,5445
4,d312,Harvey Fletcher – the Father of Stereophonic S...,3341
5,d144,The Bose-Einstein Condensate brings Quantum Th...,3262
6,d188,Encore un Moment – The Life of Madame Du Barry,4140
7,d130,Ernst Boris Chain and his Research on Antibiotics,4995
8,d036,Socrates and the Socratic Method,3256
9,d330,William Playfair and the Beginnings of Infogra...,4857


### Pre-Procesamiento.Limpieza y normalizacion

In [47]:
def clean_text(text):
    """ 
    Esta función limpia el texto de un documento eliminando etiquetas HTML y URLs.
    Además, reemplaza múltiples espacios en blanco por un solo espacio y elimina los espacios al inicio 
    y al final del texto.
    """
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_text(text, remove_accents=False):
    """ 
    Esta función normaliza el texto de un documento, 
    convirtiéndolo a minúsculas y eliminando acentos si es necesario.
    """
    text = unicodedata.normalize("NFKC", text).lower()
    if remove_accents:
        text = "".join(
            char
            for char in unicodedata.normalize("NFD", text)
            if unicodedata.category(char) != "Mn"
        )
    return text
for document in documents:
    texto = document["titulo"] + " " + document["contenido"]
    texto = clean_text(texto)
    document["contenido_limpio"] = normalize_text(texto, True)

In [48]:
datos_limpieza = []

for document in sample_documents:

    texto_original = document["contenido"]

    texto_limpio = clean_text(texto_original)
    texto_limpio = normalize_text(texto_limpio, True)

    datos_limpieza.append({
        "ID": document["id"],
        "Texto original": texto_original[:100],
        "Texto procesado": texto_limpio[:100],
        "Longitud original": len(texto_original),
        "Longitud procesada": len(texto_limpio)
    })

tabla_limpieza = pd.DataFrame(datos_limpieza)

display(tabla_limpieza)

,ID,Texto original,Texto procesado,Longitud original,Longitud procesada
0,d096,"On July 24, 1786, French geographer, astronome...","on july 24, 1786, french geographer, astronome...",4502,4502
1,d169,The frontispiece of the Diamond Sutra from Tan...,the frontispiece of the diamond sutra from tan...,4627,4627
2,d027,Giovanni Maria Lancisi (1654-1720). On October...,giovanni maria lancisi (1654-1720). on october...,3884,3886
3,d203,"On September 27, 1983, American software freed...","on september 27, 1983, american software freed...",5445,5445
4,d312,Setup for the oil drop experiment. On Septembe...,setup for the oil drop experiment. on septembe...,3341,3341
5,d144,Velocity-distribution data (3 views) for a gas...,velocity-distribution data (3 views) for a gas...,3262,3262
6,d188,"Madame Du Barry (1743 – 1793). On April 22, 1...","madame du barry (1743 – 1793). on april 22, 17...",4140,4139
7,d130,Sir Ernst Boris Chain (1906-1979). On June 19...,"sir ernst boris chain (1906-1979). on june 19,...",4995,4994
8,d036,"Socrates by Leonidas Drosis, Athens – Academy ...","socrates by leonidas drosis, athens – academy ...",3256,3252
9,d330,"Playfair’s trade-balance time-series chart, fr...","playfair’s trade-balance time-series chart, fr...",4857,4857


### Tokenizacion
Se seleccionó spaCy como tokenizer porque su segmentación permite separar términos unidos por signos de puntuación, como guiones, y conservar sus componentes como tokens independientes. Esto resulta conveniente para la recuperación booleana mediante índice invertido, ya que permite que términos individuales como jean o nicolas puedan ser indexados y recuperados independientemente.en la tabla se puede ver las evidencias de eso

In [49]:
def tokenize_nltk(text):
    return word_tokenize(text)

nlp = spacy.load("en_core_web_sm")
def tokenize_spacy(text):
    doc = nlp(text)
    return [token.text for token in doc]

In [50]:
def compare_tokens(tokens_nltk, tokens_spacy):
    """
    Esta función compara dos listas de tokens generadas por NLTK y spaCy, respectivamente.
    Devuelve dos listas:
    - Tokens que están presentes en la lista de NLTK pero no en la de spaCy.
    - Tokens que están presentes en la lista de spaCy pero no en la de NLTK.
    """
    set_nltk = set(tokens_nltk)
    set_spacy = set(tokens_spacy)

    solo_nltk = [token for token in tokens_nltk if token not in set_spacy]
    solo_spacy = [token for token in tokens_spacy if token not in set_nltk]

    return solo_nltk, solo_spacy

resultados = []

for document in documents:
    texto = document["contenido_limpio"]

    tokens_nltk = tokenize_nltk(texto)
    tokens_spacy = tokenize_spacy(texto)

    set_nltk = set(tokens_nltk)
    set_spacy = set(tokens_spacy)

    solo_nltk = [token for token in tokens_nltk if token not in set_spacy]
    solo_spacy = [token for token in tokens_spacy if token not in set_nltk]

    resultados.append({
        "id": document["id"],
        "titulo": document["titulo"],
        "tokens_nltk": len(tokens_nltk),
        "tokens_spacy": len(tokens_spacy),
        "diferencia": abs(len(tokens_nltk) - len(tokens_spacy)),
        "solo_NLTK": solo_nltk,
        "solo_spaCy": solo_spacy
    })

comparacion = pd.DataFrame(resultados)

display(comparacion)

,id,titulo,tokens_nltk,tokens_spacy,diferencia,solo_NLTK,solo_spaCy
0,d096,Joseph Nicollet and the Upper Mississippi River,837,854,17,"[’, s, jean-nicolas, 19., pierre-simon, louis-...","[’s, jean, -, 19, -, simon, -, le, -, grand, -..."
1,d169,The very first Printed Book,878,890,12,"[’, s, so-called, didn, ’, t, ’, s, so-called,...","[’s, so, called, n’t, ’s, so, called, blocks, ..."
2,d027,Giovanni Maria Lancisi and his Medical Discove...,711,712,1,"[1654-1720, ’, s, ’, s, mosquito-infested, ’, ...","[-, ’s, ’s, mosquito, -, infested, ’s, literat..."
3,d203,GNU’s not Unix,1016,1014,2,"[s, ve, unix-like, multi-user, &, t, high-leve...","[’s, ’ve, -, like, multi, -, at&t, high, -, le..."
4,d312,Harvey Fletcher – the Father of Stereophonic S...,633,638,5,"[us-american, 1907., ph.d., s, 1916., speech-p...","[us, -, 1907, ph.d, ’s, 1916, -, produced, -, ..."
...,...,...,...,...,...,...,...
326,d280,Frederick Reines and the Neutrino,1031,1040,9,"[co-detection, s, s.a., sabbatical-in-residenc...","[co, -, ’s, s.a, sabbatical, -, -, residence, ..."
327,d268,Walter Bruch and the PAL Color Television System,736,750,14,"[olympia-kanone, olympic-cannon, ’, s, mechani...","[olympia, -, kanone, olympic, -, cannon, ’s, m..."
328,d032,Samuel W. Alderson and the Crash Test Dummies,639,648,9,"[us-american, s, sheet-metal, motor-powered, m...","[us, -, american, ’s, sheet, -, metal, motor, ..."
329,d056,The Travels of William Dampier,745,751,6,"[’, s, 1697., ‘, s, 1699., 26-gun, 1701., 26-g...","[’s, australia‘s, 1699, 26, -, gun, 26, -, gun..."


Quitamos palabras de parada y puntuacion en los tockes 

In [51]:
for document in documents:
    document["tokens"] = [ token for token in nlp(document["contenido_limpio"]) if not token.is_punct and not token.is_stop ]

In [52]:
ejemplos_limpieza = []
tabla_limpieza_tokens = []

for document in sample_documents:

    tokens_antes = [
        token
        for token in nlp(document["contenido_limpio"])
    ]

    tokens_despues = [
        token
        for token in nlp(document["contenido_limpio"])
        if not token.is_punct and not token.is_stop
    ]

    tabla_limpieza_tokens.append({
        "ID": document["id"],
        "Tokens antes": len(tokens_antes),
        "Tokens después": len(tokens_despues),
        "Tokens eliminados": len(tokens_antes) - len(tokens_despues)
    })

tabla_limpieza_tokens = pd.DataFrame(tabla_limpieza_tokens)

display(tabla_limpieza_tokens)

,ID,Tokens antes,Tokens después,Tokens eliminados
0,d096,854,396,458
1,d169,890,411,479
2,d027,712,336,376
3,d203,1014,482,532
4,d312,638,284,354
5,d144,632,303,329
6,d188,803,380,423
7,d130,947,446,501
8,d036,589,255,334
9,d330,920,431,489


### Steamming
Elegimos utilizar Porter Stemmer porque queremos que palabras que representan una misma idea, pero que aparecen en diferentes formas, puedan ser tratadas como un mismo término durante la búsqueda. Por ejemplo, engine y engines se convierten en engin, mientras que digestion y digestive se convierten en digest. Aunque el resultado no siempre sea una palabra real, esto no es un problema para nuestro motor, porque el stem se utiliza únicamente como término interno del índice. Así podemos aumentar las posibilidades de encontrar documentos relevantes aunque la consulta y el documento utilicen formas diferentes de una palabra.

Adicionalmente el proceso de stemming es menos costoso que el proceso de lematización dado que no requiere un recurso linguistico para generar el corte en la palabra.

In [53]:
for document in documents:
    document["tokens"] = [stemmer.stem(token.text) for token in document["tokens"]]

In [54]:
tabla_stemming = []

for document in sample_documents:

    tokens = document["tokens"]

    stemmed_tokens = [
        stemmer.stem(token)
        for token in tokens
    ]

    tabla_stemming.append({
        "ID": document["id"],
        "Tokens antes": len(tokens),
        "Tokens después": len(stemmed_tokens),
        "Ejemplos originales": tokens[:10],
        "Ejemplos stemmed": stemmed_tokens[:10]
    })

tabla_stemming = pd.DataFrame(tabla_stemming)

display(tabla_stemming)

,ID,Tokens antes,Tokens después,Ejemplos originales,Ejemplos stemmed
0,d096,396,396,"[joseph, nicollet, upper, mississippi, river, ...","[joseph, nicollet, upper, mississippi, river, ..."
1,d169,411,411,"[print, book, frontispiec, diamond, sutra, tan...","[print, book, frontispiec, diamond, sutra, tan..."
2,d027,336,336,"[giovanni, maria, lancisi, medic, discoveri, g...","[giovanni, maria, lancisi, medic, discoveri, g..."
3,d203,482,482,"[gnu, unix, septemb, 27, 1983, american, softw...","[gnu, unix, septemb, 27, 1983, american, softw..."
4,d312,284,284,"[harvey, fletcher, father, stereophon, sound, ...","[harvey, fletcher, father, stereophon, sound, ..."
5,d144,303,303,"[bose, einstein, condens, bring, quantum, theo...","[bose, einstein, conden, bring, quantum, theor..."
6,d188,380,380,"[encor, un, moment, life, madam, du, barri, ma...","[encor, un, moment, life, madam, du, barri, ma..."
7,d130,446,446,"[ernst, bori, chain, research, antibiot, sir, ...","[ernst, bori, chain, research, antibiot, sir, ..."
8,d036,255,255,"[socrat, socrat, method, socrat, leonida, dros...","[socrat, socrat, method, socrat, leonida, dros..."
9,d330,431,431,"[william, playfair, begin, infograph, playfair...","[william, playfair, begin, infograph, playfair..."


In [55]:
print("Número de documentos:", len(documents))
documentos_sin_tokens = [
    document["id"]
    for document in documents
    if not document["tokens"]
]

print("Documentos sin tokens:", documentos_sin_tokens)
cantidad_tokens = [
    len(document["tokens"])
    for document in documents
]

print("Mínimo:", min(cantidad_tokens))
print("Máximo:", max(cantidad_tokens))
print("Promedio:", sum(cantidad_tokens) / len(cantidad_tokens))

Número de documentos: 331
Documentos sin tokens: []
Mínimo: 125
Máximo: 701
Promedio: 342.7794561933535


## Recuperación booleana usando índice invertido (BSII).

### Creacion indice inverso

In [56]:
def build_inverted_index(documents):
    """
    Esta función construye un índice invertido a partir de una lista de documentos.
    Cada documento es un diccionario que contiene un identificador único y una lista de tokens.
    Devuelve un diccionario donde las claves son los tokens y los valores son listas de identificadores de documentos que contienen ese token.
    
    Parameters:
    documents (list): Una lista de diccionarios, donde cada diccionario representa un documento y contiene un identificador único y una lista de tokens.
    Returns:
    dict: Un diccionario que representa el índice invertido, donde las claves son los tokens y
    los valores son listas de identificadores de documentos que contienen ese token.
    """
    index = {}
    for document in documents:
        document_id = document["id"]
        for token in document["tokens"]:
            if token not in index:
                index[token] = set()
            index[token].add(document_id)
    for token in index:
        index[token] = sorted(index[token])
    return index

inverted_index = build_inverted_index(documents)
tabla_indice = pd.DataFrame(
    [
        {
            "termino": termino,
            "documentos": ", ".join(sorted(documentos))
        }
        for termino, documentos in list(inverted_index.items())[:20]
    ]
)

display(tabla_indice)

,termino,documentos
0,joseph,"d016, d019, d029, d069, d074, d084, d096, d121..."
1,nicollet,d096
2,upper,"d042, d048, d054, d096, d109, d138, d147, d172..."
3,mississippi,"d095, d096, d138, d184, d302"
4,river,"d004, d025, d029, d035, d053, d078, d090, d095..."
5,juli,"d024, d025, d071, d076, d089, d090, d091, d092..."
6,24,"d021, d025, d029, d096, d099, d125, d153, d156..."
7,1786,"d019, d086, d096, d105, d170, d289, d304, d330..."
8,french,"d001, d003, d011, d012, d014, d019, d026, d029..."
9,geograph,"d011, d029, d096, d160, d229, d239, d250, d302"


In [57]:
inverted_index.keys()

dict_keys(['joseph', 'nicollet', 'upper', 'mississippi', 'river', 'juli', '24', '1786', 'french', 'geograph', 'astronom', 'mathematician', 'nicola', 'born', 'best', 'known', 'map', 'basin', '1830', 'accur', 'time', 'provid', 'basi', 'subsequ', 'american', 'interior', 'jean', 'cluse', 'savoy', 'franc', 'bright', 'show', 'aptitud', 'mathemat', 'astronomi', 'earn', 'scholarship', 'jesuit', 'colleg', 'chamberi', 'led', 'begin', 'teach', 'age', '19', 'wish', 'educ', 'went', 'pari', 'attend', 'ecol', 'normal', 'superieur', 'taught', 'brief', 'period', '1817', 'secretari', 'librarian', 'observatori', 'continu', 'studi', 'pierr', 'simon', 'laplac', '1818', 'give', 'post', 'attach', 'royal', 'professor', 'loui', 'le', 'grand', 'work', 'discov', 'comet', 'built', 'reput', 'expert', 'physic', 'geographi', 'afterward', '1820', '1', 'rapidli', 'fine', 'teacher', 'receiv', 'legion', 'honour', 'excel', 'skill', 'appli', 'principl', 'probabl', 'stock', 'market', 'believ', 'fortun', 'consider', 'allow'

In [58]:
memoria_indice = sys.getsizeof(inverted_index)

for termino, postings in inverted_index.items():
    memoria_indice += sys.getsizeof(termino)
    memoria_indice += sys.getsizeof(postings)

    for documento in postings:
        memoria_indice += sys.getsizeof(documento)

print("Número de documentos:", len(documents))
print("Tamaño del vocabulario:", len(inverted_index))
print("Memoria aproximada del índice:", memoria_indice, "bytes")

Número de documentos: 331
Tamaño del vocabulario: 13923
Memoria aproximada del índice: 5920462 bytes


In [59]:
termino_mas_frecuente = None
termino_menos_frecuente = None

max_postings = 0
min_postings = None

for termino, postings in inverted_index.items():
    cantidad_postings = len(postings)

    if cantidad_postings > max_postings:
        max_postings = cantidad_postings
        termino_mas_frecuente = termino

    if min_postings is None or cantidad_postings < min_postings:
        min_postings = cantidad_postings
        termino_menos_frecuente = termino

print("Término más frecuente:", termino_mas_frecuente)
print("Número de postings:", max_postings)

print("Término menos frecuente:", termino_menos_frecuente)
print("Número de postings:", min_postings)

Término más frecuente: yovisto
Número de postings: 320
Término menos frecuente: nicollet
Número de postings: 1


La palabra **`yovisto`** corresponde a una referencia incluida de forma repetitiva en los textos del corpus, por lo que aparece en una gran cantidad de documentos. Como ejercicio de análisis, se eliminará temporalmente este término para observar cuánto cambia el consumo aproximado de memoria del índice invertido. Esta eliminación se realiza únicamente con fines comparativos y no modifica el procesamiento utilizado en los demás experimentos.

Para analizar el efecto del término `yovisto` sobre el índice invertido, se realiza una copia profunda de la colección original de documentos con el fin de conservar los datos originales sin modificaciones. Sobre esta copia se elimina `yovisto` de la lista de tokens de cada documento.

El índice invertido se **reconstruye completamente desde cero** utilizando los documentos filtrados. Esta decisión permite obtener una medición más representativa del tamaño real del índice sin dicho término, especialmente en el análisis de memoria, ya que eliminar directamente una entrada de un diccionario ya construido no garantiza que Python libere inmediatamente toda la memoria reservada internamente.

Sobre el nuevo índice se recalculan sus principales características: tamaño del vocabulario, memoria aproximada utilizada, término con mayor frecuencia documental, término con menor frecuencia documental y sus respectivas listas de *postings*. Los resultados obtenidos representan el comportamiento del índice como si `yovisto` no hubiera formado parte del conjunto de tokens utilizado durante su construcción.


### Analisis del indice sin el termino yovisto

In [60]:

def analizar_indice_sin_termino(documents, termino_excluir="yovisto"):

    documents_filtrados = copy.deepcopy(documents)
    for document in documents_filtrados:
        document["tokens"] = [
            token
            for token in document["tokens"]
            if token != termino_excluir
        ]
    inverted_index = build_inverted_index(documents_filtrados)
    # Tabla con los primeros 20 términos
    tabla_indice = pd.DataFrame(
        [
            {
                "termino": termino,
                "documentos": ", ".join(documentos),
                "num_documentos": len(documentos),
            }
            for termino, documentos in list(inverted_index.items())[:20]
        ]
    )
    display(tabla_indice)
    memoria_indice = sys.getsizeof(inverted_index)
    for termino, postings in inverted_index.items():
        memoria_indice += sys.getsizeof(termino)
        memoria_indice += sys.getsizeof(postings)

        for documento in postings:
            memoria_indice += sys.getsizeof(documento)

    termino_mas_frecuente = max(inverted_index,key=lambda termino: len(inverted_index[termino]))
    termino_menos_frecuente = min(inverted_index,key=lambda termino: len(inverted_index[termino]))


    print("Término eliminado:", termino_excluir)
    print("Número de documentos:", len(documents_filtrados))
    print("Tamaño del vocabulario:", len(inverted_index))
    print("Memoria aproximada del índice:", memoria_indice, "bytes")
    print("Término más frecuente:",termino_mas_frecuente)
    print("Número de postings:",len(inverted_index[termino_mas_frecuente]))
    print("Término menos frecuente:",termino_menos_frecuente)
    print("Número de postings:",len(inverted_index[termino_menos_frecuente]))
    return documents_filtrados, inverted_index, tabla_indice

In [61]:
documents_sin_yovisto, inverted_index_sin_yovisto, tabla_indice_sin_yovisto = (
    analizar_indice_sin_termino(
        documents,
        termino_excluir="yovisto"
    )
)

,termino,documentos,num_documentos
0,joseph,"d016, d019, d029, d069, d074, d084, d096, d121...",28
1,nicollet,d096,1
2,upper,"d042, d048, d054, d096, d109, d138, d147, d172...",14
3,mississippi,"d095, d096, d138, d184, d302",5
4,river,"d004, d025, d029, d035, d053, d078, d090, d095...",26
5,juli,"d024, d025, d071, d076, d089, d090, d091, d092...",59
6,24,"d021, d025, d029, d096, d099, d125, d153, d156...",20
7,1786,"d019, d086, d096, d105, d170, d289, d304, d330...",9
8,french,"d001, d003, d011, d012, d014, d019, d026, d029...",75
9,geograph,"d011, d029, d096, d160, d229, d239, d250, d302",8


Término eliminado: yovisto
Número de documentos: 331
Tamaño del vocabulario: 13922
Memoria aproximada del índice: 5903398 bytes
Término más frecuente: born
Número de postings: 271
Término menos frecuente: nicollet
Número de postings: 1


In [62]:
diferencia = abs(6216254 - 6199190)

porcentaje = (diferencia / 6216254) * 100

print("Diferencia:", diferencia, "bytes")
print("Porcentaje de reducción:", porcentaje, "%")

Diferencia: 17064 bytes
Porcentaje de reducción: 0.27450615756692054 %


### Consultas booleanas


Sin *skip pointers*, la intersección tiene complejidad $O(m+n)$ ya que el algortimo que se usa es un *merge* de las listas de postings asociadas a los terminos. Con *skip pointers* se pueden reducir las comparaciones  procesando las intersecciones en menos tiempo al saltar segmentos que no pueden producir coincidencias; sin embargo, el beneficio depende de la distribución y longitud de las listas, por lo que no debe asumirse una mejora asintótica fija para todos los casos. En este caso, tomando $\sqrt{p}$ *skip pointers* distribuidos uniformemente, en el mejor de los casos la complejidad se reduce a:


$$
O(\sqrt{m} + \sqrt{n})
$$


Los skip pointers no son útiles para las consultas OR, ya que en este caso no buscamos únicamente documentos que aparezcan en ambas listas, sino que necesitamos conservar todos los documentos presentes en cualquiera de ellas, eliminando solamente los repetidos. Por esta razón, saltar posiciones mediante skip pointers podría hacer que se omitan documentos que deberían formar parte del resultado.


In [63]:
def build_skip_pointers(postings):
    """
    Construye punteros de salto para una lista de postings.

    Los punteros de salto aprovechan la estructura ordenada de los
    postings para permitir saltos durante la intersección de listas,
    mejorando la eficiencia de las búsquedas.

    La longitud del salto se determina utilizando la raíz cuadrada
    del tamaño de la lista de postings.

    Parameters:
        postings (list): Lista ordenada de identificadores de documentos.

    Returns:
        dict: Diccionario donde las claves son los índices de los
            postings y los valores son los índices de destino de los saltos.
    """
    skips = {}
    skip_length = int(math.sqrt(len(postings)))
    if skip_length < 2:
        return skips

    for i in range(0, len(postings) - skip_length, skip_length):
        skips[i] = i + skip_length

    return skips


def intersect_without_skips(postings1, postings2):
    """ 
    Compara dos lista: postings1 y postings2 a partir de punteros mientras existan elementos que comparar en ambas listas. 
    Iteración i y j sean menores a la cantidad de obtenos dentro del posting 1 y 2 respectivamente. 

    La función va contando las comparaciones y consolidando las intersecciones en result.

    Parameters:
        postings1: Lista de documentos ordenados en los que aparece un termino dado 
        postings2: Lista de documentos ordenados en los que aparece el termino comparable 

    Returns:
        result: Lista de documentos ordenados en los que aparecen ambos terminos
        comparisons: Número de comparaciones realizadas para evaluar el consumo sin punteros de salto.   

    """
    result = []
    comparisons = 0
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        comparisons += 1
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            i += 1
        else:
            j += 1
    return result, comparisons

def union_postings(postings1, postings2):
    """
    Esta función une los documentos de los postings garantizando la no duplicidad de los mismos.
    Sirve para responder querys de OR. En donde se espera el retorno de documentos con al menos uno de los terminos consultados.


    Parameters:
        postings1: Lista de documentos ordenados en los que aparece un termino dado 
        postings2: Lista de documentos ordenados en los que aparece el termino comparable 
        
    Returns:
        result: Lista de todos documentos ordenados sin duplicados.
    """
    result = []
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            result.append(postings1[i])
            i += 1
        else:
            result.append(postings2[j])
            j += 1
    while i < len(postings1):
        result.append(postings1[i])
        i += 1

    while j < len(postings2):
        result.append(postings2[j])
        j += 1

    return result

def not_postings(postings, documents):
    """
    Esta función busca entregar los documentos que no contienen al termino.
    
    Parameters:
        postings: Lista de documentos ordenados en los que aparece un termino dado.
        documents: Lista de documentos del corpus
    
    Returns:
        result: Lista de documenos ordenados que no contienen el termino.

    """
    result = []
    documentos_postings = set(postings)
    for document in documents:
        if document["id"] not in documentos_postings:
            result.append(document["id"])
    result.sort()
    return result

def intersect_with_skips(postings1, postings2, skips1, skips2):
    """Calcula la intersección de dos listas de postings ordenadas utilizando skip pointers para mejorar la eficiencia de la búsqueda.

    Durante la intersección, compara los documentos de ambas listas mediante dos punteros.
    Cuando el documento de una lista es menor que el de la otra, intenta utilizar un skip pointer para avanzar varias posiciones.
    El salto solo se realiza si su documento de destino no supera el documento actual de la otra lista, evitando así omitir posibles coincidencias.

    Parameters:
        postings1: Primera lista de identificadores de documentos, ordenada de forma ascendente.
        postings2: Segunda lista de identificadores de documentos, ordenada de forma ascendente.
        skips1: Diccionario de skip pointers para postings1. Las claves son índices de la lista y los valores son los índices de destino del salto.
        skips2: Diccionario de skip pointers para postings2. Las claves son índices de la lista y los valores son los índices de destino del salto.

    Returns:
        tuple: Una tupla con tres elementos:
            - result (list): Lista ordenada de documentos presentes en ambas posting lists.
            - comparisons (int): Número de comparaciones realizadas durante la intersección.
            - skips_used (int): Número de skip pointers utilizados."""
    result = []
    comparisons = 0
    skips_used = 0
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        comparisons += 1
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            if (i in skips1 and postings1[skips1[i]] <= postings2[j]):
                i = skips1[i]
                skips_used += 1
            else:
                i += 1
        else:
            if (j in skips2 and postings2[skips2[j]] <= postings1[i]):
                j = skips2[j]
                skips_used += 1
            else:
                j += 1
    return result, comparisons, skips_used

### Evaluacion Queries

#### Consulta booleana

In [64]:
def split_boolean_query(query):
    """
    Divide una consulta booleana en tokens individuales, separando explícitamente los paréntesis para facilitar su posterior procesamiento.

    Parameters:
        query (str): Consulta booleana ingresada como una cadena de texto. Puede contener términos, operadores booleanos (AND, OR, NOT) y paréntesis.

    Returns:
        list: Lista de tokens obtenidos a partir de la consulta."""
    query = query.replace("(", " ( ")
    query = query.replace(")", " ) ")

    return query.split()

def process_boolean_tokens(tokens):
    """Preprocesa los tokens de una consulta booleana antes de evaluarla.

    Los operadores booleanos y los paréntesis se conservan sin modificar.
    Los términos de búsqueda se limpian, normalizan, tokenizan, filtran
    para eliminar signos de puntuación y palabras vacías, y finalmente
    se reducen mediante stemming para que sean compatibles con los
    términos utilizados en el índice invertido.

    Parameters:
        tokens (list): Lista de tokens obtenidos a partir de una consulta
            booleana.

    Returns:
        list: Lista de tokens procesados, conservando los operadores
            booleanos y paréntesis."""
    
    result = []
    operators = ["AND", "OR", "NOT", "(", ")"]
    for token in tokens:
        if token in operators:
            result.append(token)
        else:
            content = clean_text(token)
            contentN = normalize_text(content, True)
            processed_tokens = [ t.text for t in nlp(contentN) if not t.is_punct and not t.is_stop]
            for processed_token in processed_tokens:result.append(stemmer.stem(processed_token))
    return result

def parse_not(tokens, position):
    """Evalúa expresiones que contienen operadores NOT, términos individuales
    o expresiones agrupadas mediante paréntesis.

    El operador NOT se evalúa de forma recursiva y obtiene el complemento
    de la posting list correspondiente respecto al conjunto de documentos.
    Cuando encuentra un paréntesis, delega la evaluación de la expresión
    interna al parser de OR.

    Parameters:
        tokens (list): Lista de tokens de la consulta booleana procesada.
        position (int): Posición actual dentro de la lista de tokens.

    Returns:
        tuple: Una tupla formada por:
            - result (list): Posting list resultante de la expresión evaluada.
            - position (int): Posición del siguiente token que debe ser
              procesado."""
    
    if tokens[position] == "NOT":
        result, position = parse_not(tokens, position + 1)
        return not_postings(result, documents), position
    if tokens[position] == "(":
        result, position = parse_or(tokens, position + 1)
        return result, position + 1
    return inverted_index.get(tokens[position], []), position + 1

def parse_and(tokens, position):
    """Evalúa expresiones booleanas que contienen el operador AND.

    Obtiene inicialmente una expresión mediante parse_not y posteriormente
    realiza intersecciones sucesivas cuando encuentra operadores AND.
    Para cada intersección construye skip pointers para ambas posting lists
    y utiliza dichos punteros para mejorar la eficiencia de la intersección.

    Parameters:
        tokens (list): Lista de tokens de la consulta booleana procesada.
        position (int): Posición actual dentro de la lista de tokens.

    Returns:
        tuple: Una tupla formada por:
            - result (list): Posting list resultante de la intersección.
            - position (int): Posición del siguiente token que debe ser
              procesado."""
    
    result, position = parse_not(tokens, position)
    while position < len(tokens) and tokens[position] == "AND":
        next_result, position = parse_not(tokens, position + 1)
        skips1 = build_skip_pointers(result)
        skips2 = build_skip_pointers(next_result)
        result, _, _ = intersect_with_skips(result, next_result, skips1, skips2)
    return result, position

def parse_or(tokens, position):
    """Evalúa expresiones booleanas que contienen el operador OR.

    Obtiene inicialmente una expresión mediante parse_and y posteriormente
    realiza uniones sucesivas cuando encuentra operadores OR. La función
    delega la evaluación de las expresiones AND a parse_and, estableciendo
    así la precedencia de AND sobre OR.

    Parameters:
        tokens (list): Lista de tokens de la consulta booleana procesada.
        position (int): Posición actual dentro de la lista de tokens.

    Returns:
        tuple: Una tupla formada por:
            - result (list): Posting list resultante de la unión.
            - position (int): Posición del siguiente token que debe ser
              procesado."""
    result, position = parse_and(tokens, position)
    while position < len(tokens) and tokens[position] == "OR":
        next_result, position = parse_and(tokens, position + 1)
        result = union_postings(result, next_result)
    return result, position

def evaluate_boolean_query(tokens):
    """Evalúa una consulta booleana completa a partir de sus tokens procesados.

    Inicia el análisis sintáctico desde la primera posición de la consulta
    utilizando parse_or, que a su vez coordina la evaluación de las
    expresiones AND, OR, NOT y los paréntesis.

    Parameters:
        tokens (list): Lista de tokens de una consulta booleana procesada.

    Returns:
        list: Lista ordenada de identificadores de los documentos que
            satisfacen la consulta booleana."""
    result, _ = parse_or(tokens, 0)
    return result

In [65]:

process_boolean_tokens(split_boolean_query("river AND (french OR method)"))

['river', 'AND', '(', 'french', 'OR', 'method', ')']

In [66]:
pruebas = [
    "river AND french",
    "river OR french",
    "NOT river",
    "river AND NOT french",
    "river OR NOT french",
    "river OR french AND method",
    "river AND (french OR method)",
    "(river OR french) AND method"
]

for query in pruebas:
    tokens = split_boolean_query(query)
    tokens = process_boolean_tokens(tokens)

    resultado = evaluate_boolean_query(tokens)

    print(query)
    print("Tokens:", tokens)
    print("Cantidad:", len(resultado))
    print("Resultado:", resultado)
    print()

river AND french
Tokens: ['river', 'AND', 'french']
Cantidad: 7
Resultado: ['d029', 'd053', 'd096', 'd184', 'd189', 'd257', 'd313']

river OR french
Tokens: ['river', 'OR', 'french']
Cantidad: 94
Resultado: ['d001', 'd003', 'd004', 'd011', 'd012', 'd014', 'd019', 'd025', 'd026', 'd029', 'd035', 'd038', 'd042', 'd047', 'd052', 'd053', 'd056', 'd063', 'd066', 'd074', 'd075', 'd078', 'd083', 'd085', 'd090', 'd095', 'd096', 'd099', 'd101', 'd105', 'd108', 'd109', 'd111', 'd120', 'd122', 'd126', 'd135', 'd138', 'd139', 'd140', 'd141', 'd145', 'd147', 'd148', 'd150', 'd152', 'd157', 'd160', 'd162', 'd164', 'd166', 'd167', 'd168', 'd170', 'd181', 'd184', 'd185', 'd188', 'd189', 'd190', 'd191', 'd202', 'd212', 'd231', 'd232', 'd235', 'd236', 'd237', 'd242', 'd246', 'd248', 'd252', 'd257', 'd263', 'd264', 'd265', 'd268', 'd280', 'd281', 'd283', 'd285', 'd288', 'd293', 'd302', 'd304', 'd309', 'd313', 'd317', 'd323', 'd324', 'd325', 'd327', 'd330', 'd331']

NOT river
Tokens: ['NOT', 'river']
Cant

####  ¿Cómo se compara el algoritmo de mezcla con y sin skip pointers para la consulta early AND telecommunication?

Para la consulta **`early AND telecommunication`**, ambos algoritmos recuperan los mismos documentos: `['d060', 'd100', 'd231']`. Sin embargo, el algoritmo **sin skip pointers** realiza **107 comparaciones**, mientras que el algoritmo **con skip pointers** realiza únicamente **41 comparaciones**, utilizando **6 saltos**. Esto representa una reducción de aproximadamente **61.7 % en el número de comparaciones**, mostrando que los *skip pointers* permiten evitar recorrer posiciones innecesarias de las listas de postings y hacen más eficiente la intersección en este caso.


In [67]:
term1 = stemmer.stem("early")
term2 = stemmer.stem("telecommunication")
p1 = inverted_index.get(term1, [])
p2 = inverted_index.get(term2, [])

print("early:", p1)
print("telecommunication:", p2)

print(term1)
print(term2)

print("early:", p1)
print("telecommunication:", p2)

early: ['d001', 'd003', 'd009', 'd014', 'd015', 'd016', 'd017', 'd018', 'd021', 'd022', 'd023', 'd024', 'd025', 'd027', 'd029', 'd034', 'd035', 'd039', 'd045', 'd046', 'd048', 'd052', 'd054', 'd055', 'd056', 'd057', 'd058', 'd060', 'd061', 'd063', 'd065', 'd066', 'd068', 'd069', 'd071', 'd073', 'd074', 'd076', 'd077', 'd080', 'd081', 'd085', 'd086', 'd091', 'd093', 'd095', 'd097', 'd100', 'd101', 'd107', 'd109', 'd110', 'd113', 'd115', 'd118', 'd122', 'd124', 'd126', 'd129', 'd130', 'd131', 'd132', 'd133', 'd135', 'd136', 'd137', 'd138', 'd141', 'd142', 'd144', 'd146', 'd148', 'd151', 'd152', 'd154', 'd159', 'd167', 'd168', 'd171', 'd172', 'd173', 'd174', 'd175', 'd185', 'd190', 'd192', 'd193', 'd194', 'd198', 'd199', 'd201', 'd203', 'd204', 'd205', 'd209', 'd211', 'd212', 'd214', 'd215', 'd216', 'd218', 'd219', 'd221', 'd223', 'd229', 'd230', 'd231', 'd232', 'd233', 'd234', 'd235', 'd237', 'd240', 'd241', 'd244', 'd247', 'd248', 'd249', 'd250', 'd251', 'd255', 'd257', 'd259', 'd262', 

In [68]:
resultado_sin, comparaciones_sin = intersect_without_skips(
    p1,
    p2
)

skips_p1 = build_skip_pointers(p1)
skips_p2 = build_skip_pointers(p2)

resultado_con, comparaciones_con, skips_utilizados = intersect_with_skips(
    p1,
    p2,
    skips_p1,
    skips_p2
)

print("Resultado sin skips:", resultado_sin)
print("Resultado con skips:", resultado_con)

print("Comparaciones sin skips:", comparaciones_sin)
print("Comparaciones con skips:", comparaciones_con)

print("Skip pointers utilizados:", skips_utilizados)

print("Reducción al usar skips:", ((107 - 41) / 107) * 100, "%")

Resultado sin skips: ['d060', 'd100', 'd231']
Resultado con skips: ['d060', 'd100', 'd231']
Comparaciones sin skips: 107
Comparaciones con skips: 41
Skip pointers utilizados: 6
Reducción al usar skips: 61.6822429906542 %


#### Consultas binarias AND

In [69]:
queries_path = Path("queries-raw-texts")
query_files = list(queries_path.glob("*.naf"))

queries = []

for file_path in query_files:
    query = read_naf_document(file_path)
    queries.append(query)

In [70]:
queries[:5]

[{'id': 'q16', 'contenido': 'South America'},
 {'id': 'q25', 'contenido': 'WWII aircraft'},
 {'id': 'q19', 'contenido': 'William Hearst movie'},
 {'id': 'q03', 'contenido': 'Romanticism'},
 {'id': 'q01', 'contenido': 'Fabrication of music instruments'}]

In [71]:
for query in queries:

    content = clean_text(query["contenido"])
    contentN = normalize_text( content,True)
    tokens = [
        token.text
        for token in nlp(contentN)
        if not token.is_punct and not token.is_stop
    ]
    query["stems"] = [ stemmer.stem(token) for token in tokens]

In [72]:
resultados_queries = []

for query in queries:

    terms = query["stems"]

    if not terms:
        resultados_queries.append({
            "query": query["id"],
            "terminos": "",
            "resultado": [],
            "comparaciones_sin_skips": 0,
            "comparaciones_con_skips": 0,
            "skips_utilizados": 0,
            "iguales": True
        })
        continue

    resultado_sin = inverted_index.get(terms[0], [])
    comparaciones_sin = 0

    for term in terms[1:]:
        postings = inverted_index.get(term, [])
        resultado_sin, comparaciones = intersect_without_skips(
            resultado_sin,
            postings
        )
        comparaciones_sin += comparaciones

        if not resultado_sin:
            break

    resultado_con = inverted_index.get(terms[0], [])
    comparaciones_con = 0
    skips_utilizados = 0

    for term in terms[1:]:

        postings = inverted_index.get(term, [])

        skips_resultado = build_skip_pointers(resultado_con)
        skips_postings = build_skip_pointers(postings)

        resultado_con, comparaciones, skips = intersect_with_skips(
            resultado_con,
            postings,
            skips_resultado,
            skips_postings
        )

        comparaciones_con += comparaciones
        skips_utilizados += skips

        if not resultado_con:
            break

    resultados_queries.append({
        "query": query["id"],
        "terminos": " AND ".join(terms),
        "resultado": resultado_con,
        "comparaciones_sin_skips": comparaciones_sin,
        "comparaciones_con_skips": comparaciones_con,
        "skips_utilizados": skips_utilizados,
        "iguales": resultado_sin == resultado_con
    })

tabla_queries = pd.DataFrame(resultados_queries)

display(tabla_queries)

,query,terminos,resultado,comparaciones_sin_skips,comparaciones_con_skips,skips_utilizados,iguales
0,q16,south AND america,"[d132, d150, d176, d184, d229, d250, d277]",46,46,0,True
1,q25,wwii AND aircraft,[],0,0,0,True
2,q19,william AND hearst AND movi,[d179],31,10,4,True
3,q03,romantic,"[d105, d147, d152, d283, d291, d318]",0,0,0,True
4,q01,fabric AND music AND instrument,[],34,26,2,True
5,q23,grace AND hopper AND famou,[d219],120,32,8,True
6,q26,literari AND critic AND thoma AND moor,[],108,94,4,True
7,q32,roman AND empir,"[d025, d031, d090, d139, d254]",36,36,0,True
8,q44,napoleon AND russian AND campaign,"[d029, d185]",40,38,1,True
9,q45,friend AND enemi AND napoleon AND bonapart,[d105],76,55,4,True


In [73]:
tabla_queries_ordenada = tabla_queries.copy()
numeros = []

for query_id in tabla_queries_ordenada["query"]:
    numeros.append(int(query_id[1:]))

tabla_queries_ordenada["numero_query"] = numeros
tabla_queries_ordenada = tabla_queries_ordenada.sort_values("numero_query")
output_path = Path("BSII-AND-queries_results")

with output_path.open("w", encoding="utf-8") as output_file:
    for _, row in tabla_queries_ordenada.iterrows():
        documents_found = sorted(row["resultado"])
        output_file.write(
            f'{row["query"]} {",".join(documents_found)}\n'
        )

print(f"Archivo generado: {output_path.resolve()}")

Archivo generado: /home/lorena/python/NLP-ml/Tarea1/BSII-AND-queries_results


## Recuperación ranqueada y vectorización de documentos (RRDV)

### Representación vectorial ponderada tf.idf

La estrategia utiliza el índice invertido para obtener el DF de cada término y limitar el cálculo del TF a los documentos en los que el término aparece. Esto evita calcular el TF para documentos en los que el término no está presente y aprovecha la estructura construida para BSII. Sin embargo, el índice invertido almacena únicamente los identificadores de los documentos, por lo que la frecuencia del término debe calcularse nuevamente a partir de los tokens de cada documento. Además, para localizar cada documento asociado a un posting, la implementación recorre la colección de documentos. Por esta razón, aunque se reduce el número de documentos sobre los que se calcula el TF, la implementación no es completamente eficiente

In [74]:
def build_tfidf_matrix(inverted_index, documents):
    """Construye la matriz TF-IDF utilizando el índice invertido para obtener
    el DF y calcular el peso de cada término en los documentos donde aparece.
    
    Parameters:
        inverted_index: Diccionario del Índice invertido en donde relaciona cada término con la lista de identificadores de los documentos en los que aparece.
        documents: Lista de documentos que contiene sus identificadores y tokens, utilizados para calcular la frecuencia del término (TF).
    Returns:
        tf_idf_matriz: Matriz TF-IDF representada como un diccionario donde cada término contiene los documentos en los que aparece y su respectivo peso TF-IDF.
    """
    tf_idf_matrix = {}

    for term in inverted_index:
        tf_idf_matrix[term] = {}
        df = len(inverted_index[term])
        idf = np.log10(len(documents) / df)

        for document_id in inverted_index[term]:
            for document in documents:
                if document["id"] == document_id:
                    tf = document["tokens"].count(term)
                    tf_idf_matrix[term][document_id] = np.log10(1 + tf) * idf
                    break

    return tf_idf_matrix     

 
tf_idf_matrix = build_tfidf_matrix(inverted_index, documents)

In [75]:
def build_document_vectors(tf_idf_matrix, documents):
    """
    Construye un vector TF-IDF para cada documento a partir de la matriz TF-IDF, asignando cero a los términos que no aparecen en el documento.

    Parameters:
        tf_idf_matrix: Diccionario de la Matriz TF-IDF representada como un diccionario
            de términos, documentos y sus respectivos pesos.
        documents: Lista de documentos con sus identificadores.

    Returns:
        dict: Diccionario que relaciona cada identificador de documento con su vector TF-IDF.
    """
    document_vectors = {}

    for document in documents:
        document_id = document["id"]
        vector = []
        for term in tf_idf_matrix:
            vector.append(tf_idf_matrix[term].get(document_id, 0))
        document_vectors[document_id] = vector

    return document_vectors

document_vectors = build_document_vectors(tf_idf_matrix,documents)

def build_query_vector(query, tf_idf_matrix, inverted_index, documents):
    """Construye el vector TF-IDF de una consulta utilizando el mismo
    vocabulario y esquema de ponderación empleado para los documentos.

    Parameters:
        query: Diccionario de la consulta procesada que contiene los términos normalizados o stemmizados.
        tf_idf_matrix: Diccionario Matriz TF-IDF utilizada para definir el vocabulario y los pesos de los términos.
        inverted_index: Diccionario Índice invertido utilizado para obtener el DF de cada término de la consulta.
        documents : Lista de documentos utilizada para calcular el IDF.

    Returns:
        list: Vector TF-IDF correspondiente a la consulta."""
    
    query_vector = []
    for term in tf_idf_matrix:
        tf = query["stems"].count(term)
        if tf > 0:
            df = len(inverted_index[term])
            idf = np.log10(len(documents) / df)
            tf_idf = np.log10(1 + tf) * idf
        else:
            tf_idf = 0
        query_vector.append(tf_idf)
    return query_vector
    

# vector ara uno de los queries
query_vector = build_query_vector(
    query,
    tf_idf_matrix,
    inverted_index,
    documents
)
print("Dimensión:", len(query_vector))
print("Primeros 10 valores:", query_vector[:10])

    

Dimensión: 13923
Primeros 10 valores: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [76]:

ids = list(document_vectors.keys())[:10]

# Terminos que tienen algun peso en esos documentos
terms = []

for term in tf_idf_matrix:
    if any(tf_idf_matrix[term].get(doc_id, 0) > 0 for doc_id in ids):
        terms.append(term)

# Construimos la tabla
vector_table = pd.DataFrame(
    {
        term: [tf_idf_matrix[term].get(doc_id, 0) for doc_id in ids]
        for term in terms
    },
    index=ids
)

vector_table = vector_table.T
display(vector_table)

,d096,d169,d027,d203,d312,d144,d188,d130,d036,d330
joseph,0.645812,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.322906
nicollet,3.222239,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
upper,0.960175,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
mississippi,1.965036,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
river,1.266305,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...
talk,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.237041
rosl,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.667925
let,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.343059
dataset,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.667925


### Similitud del coseno

In [77]:
def cosine_similarity(vector1, vector2):
    """Calcula la similitud coseno entre dos vectores y retorna un valor que representa su similitud.
    
    Parameters:
        vector1: Lista primer vector numérico.
        vector2: Lista segundo vector numérico.

    Returns:
        float: Similitud coseno entre los dos vectores. Retorna 0 si
            alguno de los vectores tiene norma cero."""
    dot_product = np.dot(vector1, vector2)
    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)

    if norm1 == 0 or norm2 == 0:
        return 0

    return dot_product / (norm1 * norm2)

d1 = document_vectors["d096"]
d2 = document_vectors["d169"]

similarity = cosine_similarity(d1, d2)

print("Similitud:", similarity)

Similitud: 0.01940197047531942


In [78]:
def rank_documents(query,document_vectors,tf_idf_matrix,inverted_index,documents):
    """
    Ordena los documentos según su similitud coseno con una consulta, colocando primero los documentos con mayor similitud.

    Parameters:
        query (dict): Consulta procesada que contiene sus términos.
        document_vectors (dict): Vectores TF-IDF de los documentos.
        tf_idf_matrix (dict): Matriz TF-IDF utilizada para construir el vector de la consulta.
        inverted_index (dict): Índice invertido utilizado para obtener el DF de los términos.
        documents (list): Lista de documentos utilizada para calcular el IDF.

    Returns:
        list: Lista de tuplas con el identificador del documento y su
            similitud con la consulta, ordenada de mayor a menor similitud.
    """
    query_vector = build_query_vector(query,tf_idf_matrix,inverted_index,documents)
    rankings = []

    for document_id, document_vector in document_vectors.items():

        similarity = cosine_similarity(query_vector,document_vector)

        if similarity > 0:
            rankings.append((document_id, similarity))

    return sorted(rankings,key=lambda x: x[1],reverse=True)

In [79]:
coseno_rankings = {}

for query in queries:
    coseno_rankings[query["id"]] = rank_documents(query,document_vectors,tf_idf_matrix,inverted_index,documents)

In [80]:
filas = []

for query_id, ranking in coseno_rankings.items():

    for posicion, (document_id, similarity) in enumerate(ranking[:10], start=1):

        filas.append({
            "query": query_id,
            "posición": posicion,
            "documento": document_id,
            "similitud": similarity
        })

tabla_ranking = pd.DataFrame(filas)

display(tabla_ranking)

,query,posición,documento,similitud
0,q16,1,d184,0.121080
1,q16,2,d132,0.119054
2,q16,3,d250,0.097747
3,q16,4,d176,0.094663
4,q16,5,d277,0.093061
...,...,...,...,...
341,q29,6,d120,0.066356
342,q29,7,d133,0.065400
343,q29,8,d046,0.049038
344,q29,9,d062,0.048644


In [81]:
query_id = "q06"

tabla_q06 = pd.DataFrame(
    coseno_rankings[query_id][:10],
    columns=["documento", "similitud"]
)

tabla_q06.insert(0, "posición", range(1, len(tabla_q06) + 1))

display(tabla_q06)

,posición,documento,similitud
0,1,d329,0.230026
1,2,d297,0.224940
2,3,d026,0.160921
3,4,d029,0.120839
4,5,d233,0.106594
5,6,d025,0.104681
6,7,d257,0.103213
7,8,d069,0.065687
8,9,d186,0.045624
9,10,d077,0.043958


In [82]:
ids = ["d329", "d297", "d026","d029","d233"]

for document in documents:
    if document["id"] in ids:
        print("=" * 80)
        print("ID:", document["id"])
        print("Título:", document["titulo"])
        print("Contenido:", document["contenido_limpio"])

ID: d026
Título: Jean-Rondolphe Perronet and the Bridges of Paris
Contenido: jean-rondolphe perronet and the bridges of paris jean-rodolphe perronet (1708-1794). on october 27, 1708, french architect and structural engineer jean-rodolphe perronet was born. he is best known for his many stone arch bridges, among them his most popular work, the paris pont de la concorde. jean-rodolphe perronet was born in suresnes, a suburb of paris, the son of a swiss guardsman. at 17 he entered the architectural practice of jean beausire, “first architect” to the city of paris, as an apprentice. he was put in charge of the design and construction of paris’s grand sewer, embankment works and the maintenance of the banlieue’s roads. in 1735, he was named sous-ingenieur (under-engineer) to alencon. perronet’s perceived energy for the district of alencon came to the notice of trudaine – the overseer of finances in charge of roads – who put him in charge of training surveyors and those drawing maps to provi

### BM25

In [83]:
corpus = [document["tokens"] for document in documents]

bm25 = BM25Okapi(
    corpus,
    k1=1.5,
    b=0.75
)
bm25_rankings = {}
filas = []
for query in queries:
    scores = bm25.get_scores(query["stems"])
    bm25_rankings[query["id"]] = [(documents[i]["id"], scores[i]) for i in np.argsort(scores)[::-1] if scores[i] > 0]

for query_id, ranking in bm25_rankings.items():

    for posicion, (document_id, score) in enumerate(ranking[:10], start=1):

        filas.append({
            "query": query_id,
            "posición": posicion,
            "documento": document_id,
            "BM25": score
        })

tabla_bm25 = pd.DataFrame(filas)

display(tabla_bm25)

,query,posición,documento,BM25
0,q16,1,d132,8.632564
1,q16,2,d184,8.055125
2,q16,3,d176,7.657014
3,q16,4,d250,6.960854
4,q16,5,d277,6.936214
...,...,...,...,...
341,q29,6,d120,4.681754
342,q29,7,d133,4.469842
343,q29,8,d046,4.251810
344,q29,9,d062,3.599076


Al variar b entre 0 y 1 se observa que la normalización por longitud afecta el ranking de BM25. Para q06, con b=0, d297 ocupa el primer lugar y d329 el segundo; al aumentar b hasta 0.50, sus posiciones se intercambian y d329 permanece primero hasta b=1. Otros documentos, como d026, mantienen su posición durante toda la variación. Esto muestra que el efecto de b depende de las características de cada documento y que una mayor normalización por longitud puede modificar el orden de documentos con puntajes similares.

In [84]:
"Prueba de variacion de b "
query = next(q for q in queries if q["id"] == "q06")

resultados_bm25 = []

for b in [0, 0.25, 0.5, 0.75, 1]:

    bm25 = BM25Okapi(
        corpus,
        k1=1.5,
        b=b
    )

    scores = bm25.get_scores(query["stems"])
    ranking = [(documents[i]["id"], scores[i])for i in np.argsort(scores)[::-1] if scores[i] > 0]

    for posicion, (documento, score) in enumerate(ranking[:10], 1):

        resultados_bm25.append({
            "b": b,
            "posición": posicion,
            "documento": documento,
            "score": score
        })


In [85]:
tabla_bm25 = pd.DataFrame(resultados_bm25)

tabla_bm25["resultado"] = (
    tabla_bm25["documento"] 
    + " (" 
    + tabla_bm25["score"].round(2).astype(str) 
    + ")"
)

tabla_bm25 = tabla_bm25.pivot(
    index="posición",
    columns="b",
    values="resultado"
)

display(tabla_bm25)

b,0.00,0.25,0.50,0.75,1.00
posición,,,,,
1,d297 (9.54),d297 (9.62),d329 (9.72),d329 (9.83),d329 (9.96)
2,d329 (9.49),d329 (9.6),d297 (9.71),d297 (9.8),d297 (9.89)
3,d026 (9.43),d026 (9.38),d026 (9.34),d026 (9.29),d026 (9.24)
4,d257 (7.63),d257 (7.5),d257 (7.38),d257 (7.26),d029 (7.3)
5,d029 (6.99),d029 (7.06),d029 (7.14),d029 (7.22),d257 (7.15)
6,d025 (6.03),d025 (6.04),d025 (6.04),d025 (6.05),d025 (6.05)
7,d233 (5.48),d233 (5.54),d233 (5.61),d233 (5.67),d233 (5.73)
8,d069 (4.52),d069 (4.65),d069 (4.8),d069 (4.95),d069 (5.12)
9,d303 (4.52),d303 (4.15),d303 (3.84),d303 (3.57),d004 (3.49)


In [86]:
ids = ["d329", "d297", "d026","d029","d233"]

for document in documents:
    if document["id"] in ids:
        print("=" * 80)
        print("ID:", document["id"])
        print("Título:", document["titulo"])
        print("Contenido:", document["contenido_limpio"])

ID: d026
Título: Jean-Rondolphe Perronet and the Bridges of Paris
Contenido: jean-rondolphe perronet and the bridges of paris jean-rodolphe perronet (1708-1794). on october 27, 1708, french architect and structural engineer jean-rodolphe perronet was born. he is best known for his many stone arch bridges, among them his most popular work, the paris pont de la concorde. jean-rodolphe perronet was born in suresnes, a suburb of paris, the son of a swiss guardsman. at 17 he entered the architectural practice of jean beausire, “first architect” to the city of paris, as an apprentice. he was put in charge of the design and construction of paris’s grand sewer, embankment works and the maintenance of the banlieue’s roads. in 1735, he was named sous-ingenieur (under-engineer) to alencon. perronet’s perceived energy for the district of alencon came to the notice of trudaine – the overseer of finances in charge of roads – who put him in charge of training surveyors and those drawing maps to provi

### Creacion de archivos de resultados

In [87]:
with open("RRDV-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query in queries:

        resultados = rank_documents(query,document_vectors,tf_idf_matrix, inverted_index,documents)

        resultados_formateados = [ f"{document_id}: {similitud:.6f}"  for document_id, similitud in resultados ]

        archivo.write(
            f"{query['id']} {','.join(resultados_formateados)}\n"
        )

In [88]:
with open("BM25-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query_id, ranking in bm25_rankings.items():

        resultados = [f"{document_id}: {score:.6f}" for document_id, score in ranking]

        archivo.write(
            f"{query_id} {','.join(resultados)}\n"
        )

### Evaluacion de metricas (Resultados)

In [89]:
relevance_judgments = {}

with open("relevance-judgments.tsv", "r", encoding="utf-8") as archivo:
    for linea in archivo:
        query_id, documentos = linea.strip().split("\t")
        relevance_judgments[query_id] = {}
        for documento in documentos.split(","):
            document_id, relevancia = documento.split(":")
            relevance_judgments[query_id][document_id] = int(relevancia)

In [90]:
def evaluate_ranking(ranking, relevance_judgments):
    """
    Evalúa el ranking de documentos de cada consulta mediante P@M,
    R@M y NDCG@M, utilizando los juicios de relevancia correspondientes.

    Parameters:
        ranking (dict): Ranking de documentos por consulta con sus scores.
        relevance_judgments (dict): Juicios de relevancia por consulta.

    Returns:
        pd.DataFrame: Resultados de evaluación por consulta.
    """

    resultados = []

    for query_id, ranking_query in ranking.items():

        juicios = relevance_judgments.get(query_id, {})
        M = len(juicios)

        documentos = [document_id for document_id, score in ranking_query]

        relevancia_binaria = [ 1 if document_id in juicios else 0 for document_id in documentos]

        relevancia_graduada = [juicios.get(document_id, 0) for document_id in documentos]

        resultados.append({
            "query": query_id,
            "M": M,
            "P@M": metricas.precision_at_k(relevancia_binaria, M),
            "R@M": metricas.recall_at_k(relevancia_binaria, M, M),
            "NDCG@M": metricas.ndcg_at_k(
                relevancia_graduada, M, "linear"
            )
        })

    return pd.DataFrame(resultados)

#### BM25

In [91]:
resultados_bm25 = evaluate_ranking(
    bm25_rankings,
    relevance_judgments
)

display(resultados_bm25)

,query,M,P@M,R@M,NDCG@M
0,q16,2,0.500000,0.500000,0.703918
1,q25,4,0.500000,0.500000,0.674482
2,q19,2,0.500000,0.500000,1.000000
3,q03,6,1.000000,1.000000,0.989525
4,q01,3,0.666667,0.666667,0.507615
5,q23,8,0.250000,0.250000,0.593878
6,q26,1,1.000000,1.000000,1.000000
7,q32,5,1.000000,1.000000,0.996048
8,q44,10,0.700000,0.700000,0.828469
9,q45,8,0.875000,0.875000,0.937373


In [92]:
tabla_evaluacion = resultados_bm25.copy()

tabla_evaluacion[["P@M", "R@M", "NDCG@M"]] = (
    tabla_evaluacion[["P@M", "R@M", "NDCG@M"]]
    .round(4)
)

display(tabla_evaluacion)

,query,M,P@M,R@M,NDCG@M
0,q16,2,0.5000,0.5000,0.7039
1,q25,4,0.5000,0.5000,0.6745
2,q19,2,0.5000,0.5000,1.0000
3,q03,6,1.0000,1.0000,0.9895
4,q01,3,0.6667,0.6667,0.5076
5,q23,8,0.2500,0.2500,0.5939
6,q26,1,1.0000,1.0000,1.0000
7,q32,5,1.0000,1.0000,0.9960
8,q44,10,0.7000,0.7000,0.8285
9,q45,8,0.8750,0.8750,0.9374


#### Coseno

In [93]:
resultados_coseno = evaluate_ranking(
    coseno_rankings,
    relevance_judgments
)

display(resultados_coseno)

,query,M,P@M,R@M,NDCG@M
0,q16,2,0.500000,0.500000,0.444123
1,q25,4,0.500000,0.500000,0.674482
2,q19,2,0.500000,0.500000,1.000000
3,q03,6,1.000000,1.000000,0.995943
4,q01,3,0.333333,0.333333,0.196954
5,q23,8,0.250000,0.250000,0.593878
6,q26,1,1.000000,1.000000,1.000000
7,q32,5,1.000000,1.000000,0.996048
8,q44,10,0.700000,0.700000,0.803927
9,q45,8,0.750000,0.750000,0.883670


In [94]:
resultados_coseno.columns

Index(['query', 'M', 'P@M', 'R@M', 'NDCG@M'], dtype='str')

In [95]:
tabla_evaluacion_coseno = resultados_coseno.copy()

tabla_evaluacion_coseno[["P@M", "R@M", "NDCG@M"]] = (
    tabla_evaluacion_coseno[["P@M", "R@M", "NDCG@M"]]
    .round(4)
)

display(tabla_evaluacion_coseno)

,query,M,P@M,R@M,NDCG@M
0,q16,2,0.5000,0.5000,0.4441
1,q25,4,0.5000,0.5000,0.6745
2,q19,2,0.5000,0.5000,1.0000
3,q03,6,1.0000,1.0000,0.9959
4,q01,3,0.3333,0.3333,0.1970
5,q23,8,0.2500,0.2500,0.5939
6,q26,1,1.0000,1.0000,1.0000
7,q32,5,1.0000,1.0000,0.9960
8,q44,10,0.7000,0.7000,0.8039
9,q45,8,0.7500,0.7500,0.8837


In [96]:
comparacion = pd.DataFrame({
    "query": resultados_coseno["query"],
    "M": resultados_coseno["M"],
    "P@M Coseno": resultados_coseno["P@M"],
    "P@M BM25": resultados_bm25["P@M"],
    "R@M Coseno": resultados_coseno["R@M"],
    "R@M BM25": resultados_bm25["R@M"],
    "NDCG@M Coseno": resultados_coseno["NDCG@M"],
    "NDCG@M BM25": resultados_bm25["NDCG@M"]
})

display(comparacion.round(4))

,query,M,P@M Coseno,P@M BM25,R@M Coseno,R@M BM25,NDCG@M Coseno,NDCG@M BM25
0,q16,2,0.5000,0.5000,0.5000,0.5000,0.4441,0.7039
1,q25,4,0.5000,0.5000,0.5000,0.5000,0.6745,0.6745
2,q19,2,0.5000,0.5000,0.5000,0.5000,1.0000,1.0000
3,q03,6,1.0000,1.0000,1.0000,1.0000,0.9959,0.9895
4,q01,3,0.3333,0.6667,0.3333,0.6667,0.1970,0.5076
5,q23,8,0.2500,0.2500,0.2500,0.2500,0.5939,0.5939
6,q26,1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
7,q32,5,1.0000,1.0000,1.0000,1.0000,0.9960,0.9960
8,q44,10,0.7000,0.7000,0.7000,0.7000,0.8039,0.8285
9,q45,8,0.7500,0.8750,0.7500,0.8750,0.8837,0.9374


In [97]:
evaluacion_rrdv = evaluate_ranking(coseno_rankings,relevance_judgments)
evaluacion_bm25 = evaluate_ranking(bm25_rankings,relevance_judgments)

In [98]:
map_cos = metricas.mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in coseno_rankings.items()
    ]
)

map_bm25 = metricas.mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in bm25_rankings.items()
    ]
)

print("MAP COS:", map_cos)
print("MAP BM25:", map_bm25)

MAP COS: 0.7047680299424705
MAP BM25: 0.7543749373448497


## Recuperación ranqueada con GENSIM

In [99]:
corpus = [document["tokens"] for document in documents]
dictionary = corpora.Dictionary(corpus)

corpus_bow = [dictionary.doc2bow(document["tokens"])for document in documents]

print("Tamaño del vocabulario - GENSIM:", len(dictionary))

tfidf = models.TfidfModel(corpus_bow,id2word=dictionary,smartirs="ltc")
corpus_tfidf = tfidf[corpus_bow]
index = similarities.SparseMatrixSimilarity(corpus_tfidf,num_features=len(dictionary),num_best=None)

gensim_rankings = {}

for query in queries:
    query_bow = dictionary.doc2bow(query["stems"])
    query_tfidf = tfidf[query_bow]
    scores = index[query_tfidf]
    ranking = [(documents[i]["id"], scores[i]) for i in np.argsort(scores)[::-1] if scores[i] > 0]
    gensim_rankings[query["id"]] = ranking

Tamaño del vocabulario - GENSIM: 13923


In [100]:
with open("GENSIM-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query_id, ranking in gensim_rankings.items():

        resultados = [
            f"{document_id}: {score:.6f}"
            for document_id, score in ranking
        ]

        archivo.write(
            f"{query_id} {','.join(resultados)}\n"
        )

In [101]:
evaluacion_gensim = evaluate_ranking(gensim_rankings,relevance_judgments)

In [102]:
map_gensim = metricas.mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in gensim_rankings.items()
    ]
)
print("MAP GENSIM:", map_gensim)

MAP GENSIM: 0.7068766149575881


## Comparativa de metricas Finales

#### Tabla Comparativa metricas

In [103]:
evaluacion_comparativa = evaluacion_rrdv.merge(
    evaluacion_bm25,
    on=["query", "M"],
    suffixes=("_COS", "_BM25")
)

evaluacion_comparativa = evaluacion_comparativa.merge(
    evaluacion_gensim,
    on=["query", "M"]
)

evaluacion_comparativa = evaluacion_comparativa.rename(
    columns={
        "P@M": "P@M_GENSIM",
        "R@M": "R@M_GENSIM",
        "NDCG@M": "NDCG@M_GENSIM"
    }
)

evaluacion_comparativa["numero_query"] = (
    evaluacion_comparativa["query"].str[1:].astype(int)
)

evaluacion_comparativa = evaluacion_comparativa.sort_values(
    "numero_query"
).drop(columns="numero_query")

# evaluacion_comparativa = evaluacion_comparativa[
#     [
#         "query",
#         "M",
#         "P@M_COS",
#         "P@M_BM25",
#         "P@M_GENSIM",
#         "R@M_COS",
#         "R@M_BM25",
#         "R@M_GENSIM",
#         "NDCG@M_COS",
#         "NDCG@M_BM25",
#         "NDCG@M_GENSIM"
#     ]
# ]


evaluacion_comparativa = evaluacion_comparativa[
    [
        "query",
        "M",
        "P@M_GENSIM",
        "R@M_GENSIM",
        "NDCG@M_BM25",
        "NDCG@M_GENSIM"
    ]
]

display(evaluacion_comparativa.round(4))

,query,M,P@M_GENSIM,R@M_GENSIM,NDCG@M_BM25,NDCG@M_GENSIM
4,q01,3,0.3333,0.3333,0.5076,0.1970
11,q02,11,0.5455,0.5455,0.5848,0.5650
3,q03,6,1.0000,1.0000,0.9895,0.9959
31,q04,7,0.7143,0.7143,0.8910,0.7816
16,q06,6,0.8333,0.8333,0.8265,0.8497
33,q07,4,0.2500,0.2500,0.5091,0.3212
22,q08,12,0.7500,0.7500,0.8865,0.8582
32,q09,6,0.8333,0.8333,0.9179,0.8184
30,q10,8,0.3750,0.3750,0.4772,0.4683
12,q12,4,0.7500,0.7500,0.7758,0.7758


#### Tabla Comparativa MPA

In [104]:
tabla_map = pd.DataFrame({
    "Función de ranqueo": [
        "COS",
        "BM25",
        "GENSIM"
    ],
    "MAP": [
        map_cos,
        map_bm25,
        map_gensim
    ]
})

display(tabla_map)

,Función de ranqueo,MAP
0,COS,0.704768
1,BM25,0.754375
2,GENSIM,0.706877
